# 实验二 · 第三阶段：MySQL 数据存储

把第二阶段的 `data/houseinfo.csv` 写入 MySQL 的 `smartpark_lab.houseinfo` 表。

本阶段**不重新采集**，只做入库。两阶段解耦的好处是：采集逻辑改了不影响历史数据，
入库逻辑改了也不需要重新爬一遍网站。

要做的四件事：
1. 读 CSV 并校验表头
2. `executemany()` 批量插入（带事务）
3. 防重复插入（两层保险）
4. 核对入库结果

## 0. 依赖

`pymysql` 是纯 Python 的 MySQL 驱动，不需要在容器里编译。

In [1]:
%pip install pymysql

Defaulting to user installation because normal site-packages is not writeable


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


Note: you may need to restart the kernel to use updated packages.


## 1. 连接数据库

连接参数放在一个字典里，方便统一管理。`autocommit=False` 表示**不自动提交**，
事务由我们自己控制 —— 这是 `rollback()` 能生效的前提。

In [2]:
import csv
import sys
from pathlib import Path

import pymysql

DB_CONFIG = {
    "host": "mysql",
    "port": 3306,
    "user": "student",
    "password": "123456",
    "database": "smartpark_lab",
    "charset": "utf8mb4",
    "autocommit": False,
}

CSV_PATH = Path("data/houseinfo.csv")

# 8 个业务字段，顺序必须与 CSV 表头完全一致
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

conn = pymysql.connect(**DB_CONFIG)
cur = conn.cursor()
print("已连接：", DB_CONFIG["host"], "/", DB_CONFIG["database"])

cur.execute("SELECT COUNT(*) FROM houseinfo")
print("入库前条数：", cur.fetchone()[0])

cur.execute("SHOW CREATE TABLE houseinfo")
print(cur.fetchone()[1])

已连接： mysql / smartpark_lab
入库前条数： 0
CREATE TABLE `houseinfo` (
  `id` int NOT NULL AUTO_INCREMENT,
  `title` varchar(200) DEFAULT NULL,
  `house` varchar(50) DEFAULT NULL,
  `address` varchar(200) DEFAULT NULL,
  `struct` varchar(50) DEFAULT NULL,
  `area` varchar(20) DEFAULT NULL,
  `floor` varchar(20) DEFAULT NULL,
  `maketime` varchar(20) DEFAULT NULL,
  `price` varchar(20) DEFAULT NULL,
  PRIMARY KEY (`id`),
  UNIQUE KEY `uniq_houseinfo` (`title`,`house`,`address`,`struct`,`area`,`floor`,`maketime`,`price`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci


### 1.1 表结构要点

```
id        int AUTO_INCREMENT  PRIMARY KEY   <- 自增主键，插入时不能指定
title     varchar(200)
house     varchar(50)
address   varchar(200)
struct    varchar(50)      <- 户型
area      varchar(20)      <- 面积
floor     varchar(20)      <- 楼层
maketime  varchar(20)      <- 建造年代
price     varchar(20)      <- 单价
```

`id` 是自增主键，插入时**不要**在 SQL 里写它，交给数据库生成。

注意所有字段都是 `varchar`，没有数字类型 —— 因为 `area` 是 `89.9㎡`、
`price` 是 `16465元/㎡`，带单位带符号，塞不进数值列。

## 2. 建联合唯一索引（防重复的第二层保险）

表建好时**只有主键索引**，8 个业务字段上没有任何唯一约束。
这意味着如果只靠 Python 代码判断重复，并发执行或代码被绕过时，数据库会接受重复行。

给这 8 个字段建一个联合唯一索引：

```sql
CREATE UNIQUE INDEX uniq_houseinfo ON houseinfo
(title, house, address, struct, area, floor, maketime, price);
```

MySQL 没有 `CREATE UNIQUE INDEX IF NOT EXISTS`，所以先查 `information_schema`
判断索引在不在，再决定要不要建 —— 这样脚本可以重复执行而不会报错。

In [3]:
def ensure_unique_index(conn):
    """确保 8 个业务字段上有联合唯一索引。"""
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT COUNT(*) FROM information_schema.STATISTICS
            WHERE table_schema = %s AND table_name = 'houseinfo'
              AND index_name = 'uniq_houseinfo'
            """,
            (DB_CONFIG["database"],),
        )
        exists = cur.fetchone()[0]

        if exists:
            print("[索引] uniq_houseinfo 已存在，跳过创建")
            return

        cols = ", ".join(f"`{c}`" for c in FIELDS)
        cur.execute(f"CREATE UNIQUE INDEX uniq_houseinfo ON houseinfo ({cols})")
        conn.commit()
        print(f"[索引] 已创建联合唯一索引 uniq_houseinfo ({cols})")


ensure_unique_index(conn)

# 验证索引确实生效
cur.execute("SHOW INDEX FROM houseinfo")
for row in cur.fetchall():
    print("  ", row[2], "| 非唯一:", row[1], "| 列:", row[4])

[索引] uniq_houseinfo 已存在，跳过创建


   PRIMARY | 非唯一: 0 | 列: id
   uniq_houseinfo | 非唯一: 0 | 列: title
   uniq_houseinfo | 非唯一: 0 | 列: house
   uniq_houseinfo | 非唯一: 0 | 列: address
   uniq_houseinfo | 非唯一: 0 | 列: struct
   uniq_houseinfo | 非唯一: 0 | 列: area
   uniq_houseinfo | 非唯一: 0 | 列: floor
   uniq_houseinfo | 非唯一: 0 | 列: maketime
   uniq_houseinfo | 非唯一: 0 | 列: price


## 3. 读取 CSV 并校验表头

**不要假设 CSV 表头一定对。** 如果表头顺序或拼写变了，按位置取值会把
「面积」写进「楼层」列，程序不报错但数据全错。

所以读完先比对表头，不一致就抛异常停下。

In [4]:
def read_csv(path):
    """读取 CSV，返回 (记录列表, 表头)。"""
    if not path.exists():
        raise FileNotFoundError(f"找不到 CSV 文件：{path}")

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        header = reader.fieldnames
        rows = list(reader)

    if header != FIELDS:
        raise ValueError(
            f"CSV 表头与预期不一致。\n  实际：{header}\n  预期：{FIELDS}"
        )

    print(f"[读取] {path} 共 {len(rows)} 条数据，表头校验通过")
    return rows, header


rows, header = read_csv(CSV_PATH)
print("表头：", header)
print("首条：", rows[0])

[读取] data/houseinfo.csv 共 498 条数据，表头校验通过
表头： ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
首条： {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}


## 4. 批量插入（含防重复）

### 4.1 为什么用 executemany

498 条数据如果循环里逐条 `execute()`，要 498 次网络往返。
`executemany()` 把 498 组参数打包成一次发送，网络往返降到 1 次。

### 4.2 防重复的两层保险

**第一层（Python 侧）**：入库前先 SELECT 出数据库里已有的 8 字段组合，
在内存里过滤掉已存在的记录 —— 重复数据根本不会被送进数据库。

**第二层（数据库侧）**：配合唯一索引，用
`INSERT ... ON DUPLICATE KEY UPDATE`，即使第一层被绕过，
重复键也只会变成一次无害的更新，不会插入第二行。

### 4.3 事务

`try` 里 `commit()`，`except` 里 `rollback()`。
中途出错时整批回滚，不会留下「插了一半」的脏数据。

In [5]:
# ---------- 入库前条数 ----------
cur.execute("SELECT COUNT(*) FROM houseinfo")
count_before = cur.fetchone()[0]
print(f"[入库前] houseinfo 现有条数 = {count_before}")

# ---------- 第一层保险：查已有记录，在 Python 侧过滤 ----------
select_cols = ", ".join(f"`{c}`" for c in FIELDS)
cur.execute(f"SELECT {select_cols} FROM houseinfo")
existing = {tuple(r) for r in cur.fetchall()}
print(f"[查重] 数据库中已有 {len(existing)} 组不重复的记录")

to_insert = []
skipped = 0
for row in rows:
    key = tuple(row[f] for f in FIELDS)
    if key in existing:
        skipped += 1
        continue
    existing.add(key)      # 同时防止 CSV 内部自身重复
    to_insert.append(key)

print(f"[查重] CSV {len(rows)} 条中，已存在 {skipped} 条，本次需新增 {len(to_insert)} 条")

[入库前] houseinfo 现有条数 = 0
[查重] 数据库中已有 0 组不重复的记录
[查重] CSV 498 条中，已存在 0 条，本次需新增 498 条


In [6]:
# ---------- 批量插入 ----------
placeholders = ", ".join(["%s"] * len(FIELDS))
insert_sql = (
    f"INSERT INTO houseinfo ({select_cols}) VALUES ({placeholders}) "
    f"ON DUPLICATE KEY UPDATE "
    + ", ".join(f"`{c}` = VALUES(`{c}`)" for c in FIELDS)
)

try:
    if to_insert:
        cur.executemany(insert_sql, to_insert)
        conn.commit()           # 全部成功才提交
        print(f"[插入] executemany 批量插入 {len(to_insert)} 条，已 commit()")
    else:
        print("[插入] 没有需要新增的记录，跳过插入")
except Exception as exc:
    conn.rollback()             # 出错整批回滚
    print(f"[错误] {type(exc).__name__}: {exc}")
    print("[回滚] 已执行 rollback()，本次插入全部撤销")
    raise

[插入] executemany 批量插入 498 条，已 commit()


## 5. 核对入库结果

In [7]:
cur.execute("SELECT COUNT(*) FROM houseinfo")
count_after = cur.fetchone()[0]
added = count_after - count_before
print(f"[入库后] houseinfo 现有条数 = {count_after}，本次新增 = {added}")

print("\n[抽样] 前 3 条：")
cur.execute(f"SELECT id, {select_cols} FROM houseinfo ORDER BY id LIMIT 3")
for r in cur.fetchall():
    print("   ", r)

[入库后] houseinfo 现有条数 = 498，本次新增 = 498

[抽样] 前 3 条：
    (1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
    (2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
    (3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')


### 5.1 逐字段核对：数据库 vs CSV

条数相等不代表内容相等。必须把 498 条逐字段比一遍。

**注意**：这个核对要放在**插入之后**再查一次 —— 前面查的是插入前的数据，
不能拿来当结果。

In [8]:
# 从数据库重新查一遍，再与 CSV 逐字段比对
cur.execute(f"SELECT {select_cols} FROM houseinfo ORDER BY id")
db_rows = [dict(zip(FIELDS, r)) for r in cur.fetchall()]

mismatch = 0
first_diff = None
for i, (a, b) in enumerate(zip(rows, db_rows)):
    for k in FIELDS:
        if a[k] != b[k]:
            mismatch += 1
            if first_diff is None:
                first_diff = (i, k, a[k], b[k])

print("CSV 行数 =", len(rows), "| DB 行数 =", len(db_rows))
print("逐字段比对：不一致处 =", mismatch)
print("首个差异：", first_diff)

csv_set = {tuple(r[k] for k in FIELDS) for r in rows}
db_set = {tuple(r[k] for k in FIELDS) for r in db_rows}
print("DB 独有（不在 CSV）：", len(db_set - csv_set))
print("CSV 独有（不在 DB）：", len(csv_set - db_set))

for k in FIELDS:
    print(f"  {k:9s} 空值 {sum(1 for r in db_rows if not r[k])}")

CSV 行数 = 498 | DB 行数 = 498
逐字段比对：不一致处 = 0
首个差异： None
DB 独有（不在 CSV）： 0
CSV 独有（不在 DB）： 0
  title     空值 0
  house     空值 0
  address   空值 0
  struct    空值 0
  area      空值 0
  floor     空值 0
  maketime  空值 0
  price     空值 0


## 6. 独立重跑：验证不会重复插入

**关掉程序重新跑一次**，新增条数必须是 0，总条数必须还是 498。

这一步是报告里「关闭上次程序后，同批数据独立重跑的条数 / 新增条数」那一行的依据。

In [9]:
# 模拟「关闭程序后重跑」：再查一次库、看要插入几条
cur.execute(f"SELECT {select_cols} FROM houseinfo")
existing2 = {tuple(r) for r in cur.fetchall()}

to_insert2 = []
for row in rows:
    key = tuple(row[f] for f in FIELDS)
    if key not in existing2:
        to_insert2.append(key)

cur.execute("SELECT COUNT(*) FROM houseinfo")
n_now = cur.fetchone()[0]

print("重跑时数据库条数：", n_now)
print("重跑时 CSV 条数：", len(rows))
print("重跑时需新增条数：", len(to_insert2), "（应为 0）")
print("重跑后总条数：", n_now + len(to_insert2), "（应仍为 498）")

重跑时数据库条数： 498
重跑时 CSV 条数： 498
重跑时需新增条数： 0 （应为 0）
重跑后总条数： 498 （应仍为 498）


## 7. 关闭游标和连接

`finally` 块里关闭，无论成功还是失败都执行。

不关闭连接的话，多次运行脚本会累积连接，MySQL 端的连接数有上限，
累积多了会报 `Too many connections`。

In [10]:
cur.close()
conn.close()
print("[清理] 游标与连接已关闭")

[清理] 游标与连接已关闭


## 8. 汇总

In [11]:
print("===== 第三阶段汇总 =====")
print(f"CSV 数据行数：{len(rows)}")
print(f"入库前条数：{count_before}")
print(f"入库后条数：{count_after}")
print(f"本次新增条数：{added}")
print(f"逐字段比对不一致处：{mismatch}")
print(f"独立重跑新增条数：{len(to_insert2)}")
print(f"唯一索引：{select_cols} (uniq_houseinfo)")

===== 第三阶段汇总 =====
CSV 数据行数：498
入库前条数：0
入库后条数：498
本次新增条数：498
逐字段比对不一致处：0
独立重跑新增条数：0
唯一索引：`title`, `house`, `address`, `struct`, `area`, `floor`, `maketime`, `price` (uniq_houseinfo)
